# DEE v6 — SIM 21: Topología anharmónica  → ¿w(z) dinámico?**Autor:** Juan Pablo Bruschi (2026)**Versión:** v6 SIM 21**Estado:** test exploratorio del régimen anharmónico aplicado a topología cerrada T³---## Pregunta científicaEn el Capítulo 6 §6.2 del documento DEE v6, los Tests A y B mostraron que el régimen **armónico** produce w = +0,0698 universal sobre cuatro topologías (homogéneo, T³ vs R³, modos de winding, glassy). Esto es teorema derivable de la simetría de escala del Hamiltoniano cuadrático.En el Capítulo 6 §6.3 introdujimos el régimen **anharmónico** g₄·φ⁴ y obtuvimos d_S(meso) → 2,10 sobre configuración **homogénea**, lo que vía w = −d_S/3 produjo la predicción central w_DE = −0,70 ± 0,05.**Lo que no se testeó:** la combinación topología cerrada T³ + anharmonicidad. El teorema de escala armónico ya no aplica con g₄·φ⁴, y la topología cerrada inyecta una contribución extensiva (Test A original: β = +0,99) que en combinación con anharmonicidad puede:1. Producir w_top ≠ +0,0698 (rompe la universalidad armónica)2. Hacer que β dependa de g₄ → ρ_top deja de ser ≈ constante con V3. **Si β(g₄) ≠ constante, entonces w_top es dinámico** — alineable con DESI 2024 CPL evolutivo (w₀ ≈ −0,7, w_a < 0)## Hipótesis a testear| Hipótesis | Predicción β(g₄) | Implicación ||---|---|---|| H0 (armónico universal) | β → +0,99 en todos los g₄ | Topología no contribuye dinámicamente || H1 (anharmonicidad fuerte) | β(g₄) cambia significativamente | Posible w(z) dinámico || H2 (mezcla intermedia) | β(g₄) leve dependencia | Componente w_a pequeña |## Setup del test- Sustrato: red cúbica simple jiterada (jitter = 0,05) con kernel 1/d²- Topologías: T³ (BC periódicas) y R³ (BC abiertas) — mismos nodos, sólo cambia la conectividad de los bordes- Anharmonicidad: g₄ ∈ {0, 100, 1000, 5000, 10 000} (incluye g₄=0 como control que reproduce Test A)- Tamaños: N ∈ {343, 729, 1331, 2197} (4 puntos para ajuste β)- Hartree autoconsistente: igual al del Cap. 6 §6.3, con indicador χ_Hartree para validar régimen perturbativo- Observable: E_zp = ½ Σ √λ_n del K_eff post-Hartree## Criterios pre-registrados| Criterio | Valor pre-registrado | Significado ||---|---|---|| Control g₄ = 0 | β ≈ +0,99 ± 0,05 | Reproduce Test A original (validación) || Indicador Hartree | χ_Hartree < 0,5 en plateau | Régimen perturbativo válido || Sensibilidad β a g₄ | Δβ > 0,1 entre g₄=0 y g₄=10⁴ | Topología responde al régimen anharmónico || Dirección de w_top | w_top = −β cambia con g₄ | Base para w(z) dinámico |## Tiempo estimadoDiagonalización full de matrices N×N con Hartree (5–8 iteraciones) × 5 g₄ × 4 N × 2 topologías. Estimación Colab CPU: **45–80 min**.

---## 1. Setup

In [ ]:
import os, time, json
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

OUT_DIR = '/content/dee_v6_topologia'
os.makedirs(OUT_DIR, exist_ok=True)

def out_path(name): return os.path.join(OUT_DIR, name)

print(f'Outputs:  {OUT_DIR}')
print(f'NumPy:    {np.__version__}')

In [ ]:
# Paleta y estilo (consistente con notebook anterior)
BG  = '#0d1117'
CW  = '#ecf0f1'
CY  = '#f1c40f'
CG  = '#27ae60'
CB  = '#2980b9'
CR  = '#e74c3c'
CO  = '#e67e22'
CGR = '#7f8c8d'
CP  = '#9b59b6'

def setup_dark_axes(ax):
    ax.set_facecolor(BG)
    ax.tick_params(colors=CGR)
    for s in ax.spines.values():
        s.set_color('#2c3e50')

def new_figure(nrows, ncols, figsize):
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    fig.patch.set_facecolor('#0a0a1a')
    if hasattr(axes, 'flat'):
        for ax in axes.flat: setup_dark_axes(ax)
    else:
        setup_dark_axes(axes)
    return fig, axes

---## 2. Configuración globalSi querés un test más rápido, reducí `G4_VALS` (sacá los más grandes que son los más lentos) o `N_SIDES` (sacá el más grande).

In [ ]:
# ───────── parámetros del test ─────────
G4_VALS = [0.0, 100.0, 1000.0, 5000.0, 10000.0]  # 0 es control armónico
N_SIDES = [7, 9, 11, 13]                          # → N = 343, 729, 1331, 2197

# Kernel y BC
K_NEIGHBORS  = 30      # k vecinos para construir L
JITTER       = 0.05    # desorden estructural
EPS_REG      = 1e-6    # regularización para evitar autovalor 0
HARTREE_MAX  = 25      # iteraciones máximas Hartree
HARTREE_TOL  = 1e-5    # tolerancia convergencia

print(f'g₄ valores:  {G4_VALS}')
print(f'Tamaños N:   {[ns**3 for ns in N_SIDES]}')
print(f'Total puntos: {len(G4_VALS) * len(N_SIDES) * 2} (g₄ × N × topologías)')

---## 3. Funciones coreTres bloques:1. Construcción de la red jiterada (mismos nodos para ambas topologías)2. Construcción del Laplaciano: T³ con BC periódicas, R³ con BC abiertas3. Hartree autoconsistente y cálculo de E_zp

In [ ]:
def build_jittered_lattice(N_side, jitter=JITTER, seed=42):
    """Red cúbica simple jiterada en caja [0,1]³."""
    np.random.seed(seed)
    coords = np.zeros((N_side**3, 3))
    spacing = 1.0 / N_side
    idx = 0
    for i in range(N_side):
        for j in range(N_side):
            for k in range(N_side):
                base = np.array([i, j, k]) * spacing + spacing/2
                jit = spacing * jitter * np.random.uniform(-1, 1, 3)
                coords[idx] = base + jit
                idx += 1
    return coords

def pbc_distance_matrix(coords, L=1.0):
    """Distancias periódicas (imagen mínima) en cubo unitario."""
    N = len(coords)
    D2 = np.zeros((N, N))
    for dim in range(3):
        d1 = np.abs(coords[:, dim:dim+1] - coords[:, dim:dim+1].T)
        d1 = np.minimum(d1, L - d1)
        D2 += d1**2
    D = np.sqrt(D2)
    np.fill_diagonal(D, np.inf)
    return D

def open_distance_matrix(coords):
    """Distancias euclídeas estándar (BC abiertas, R³)."""
    D = cdist(coords, coords)
    np.fill_diagonal(D, np.inf)
    return D

In [ ]:
def build_laplacian(D, k_neighbors=K_NEIGHBORS):
    """Laplaciano combinatorio con k-NN y kernel 1/d²."""
    N = D.shape[0]
    W = np.zeros((N, N))
    for i in range(N):
        idx = np.argsort(D[i])[:k_neighbors]
        for j in idx:
            if D[i, j] > 0 and np.isfinite(D[i, j]):
                W[i, j] = 1.0 / D[i, j]**2
    # Simetrizar (k-NN puede dar matriz no-simétrica)
    W = np.maximum(W, W.T)
    L = np.diag(W.sum(axis=1)) - W
    return L

def hartree_self_consistent(K_base, g4, max_iter=HARTREE_MAX, tol=HARTREE_TOL,
                             eps_reg=EPS_REG, verbose=False):
    """
    Hartree autoconsistente:
      K_eff = K_base + 3·g₄·diag(⟨φ²⟩) + eps_reg·I
      ⟨φ²_i⟩ = ½ · (K_eff^(-1/2))_ii
    Devuelve K_eff convergido, autovalores de K_eff, indicador χ_Hartree,
    número de iteraciones.
    """
    N = K_base.shape[0]
    K_reg = K_base + eps_reg * np.eye(N)
    K_eff = K_reg.copy()
    phi_sq_prev = None
    chi_h = 0.0

    for it in range(max_iter):
        # Diagonalizar K_eff
        lam, vec = np.linalg.eigh(K_eff)
        lam = np.maximum(lam, eps_reg)
        # ⟨φ²_i⟩ = ½ · Σ_n |ψ_n(i)|² / √λ_n
        inv_sqrt_diag = (vec**2 @ (1.0 / np.sqrt(lam)))
        phi_sq = 0.5 * inv_sqrt_diag

        # Indicador validez Hartree (norma corrección / norma K_base)
        correction_norm = np.linalg.norm(3.0 * g4 * phi_sq)
        K_base_norm = np.linalg.norm(K_base)
        chi_h = correction_norm / max(K_base_norm, 1e-10)

        # Actualizar K_eff
        K_eff_new = K_reg + 3.0 * g4 * np.diag(phi_sq)

        # Convergencia
        if phi_sq_prev is not None:
            change = np.max(np.abs(phi_sq - phi_sq_prev) / (np.abs(phi_sq_prev) + 1e-10))
            if verbose: print(f'    iter {it+1}: change={change:.2e}, χ_H={chi_h:.4f}')
            if change < tol:
                break

        K_eff = K_eff_new
        phi_sq_prev = phi_sq.copy()

    # Diagonalización final
    lam_final = np.linalg.eigvalsh(K_eff)
    lam_final = np.maximum(lam_final, eps_reg)
    return K_eff, lam_final, chi_h, it + 1

def E_zp_from_lambda(lam):
    """Energía de punto cero del cristal cuántico: E_zp = ½ Σ √λ_n."""
    return 0.5 * np.sum(np.sqrt(np.maximum(lam, 0.0)))

---## 4. Función principal: medir ΔE_top(N, g₄)Para cada par (N, g₄):1. Construye una sola red de N nodos2. Calcula L_T³ (PBC) y L_R³ (BC abiertas) sobre la misma red3. Hartree autoconsistente para cada topología → K_eff_T³, K_eff_R³4. Calcula E_zp_T³, E_zp_R³, ΔE = E_zp_T³ − E_zp_R³Si quedó cacheado en disco lo carga sin recalcular.

In [ ]:
def medir_punto(N_side, g4, seed=42, verbose=False):
    """Devuelve dict con E_zp_T³, E_zp_R³, ΔE y diagnósticos."""
    coords = build_jittered_lattice(N_side, seed=seed)
    N = len(coords)

    # T³
    D_T3 = pbc_distance_matrix(coords)
    L_T3 = build_laplacian(D_T3)
    K_eff_T3, lam_T3, chi_T3, it_T3 = hartree_self_consistent(L_T3, g4)
    E_T3 = E_zp_from_lambda(lam_T3)

    # R³
    D_R3 = open_distance_matrix(coords)
    L_R3 = build_laplacian(D_R3)
    K_eff_R3, lam_R3, chi_R3, it_R3 = hartree_self_consistent(L_R3, g4)
    E_R3 = E_zp_from_lambda(lam_R3)

    dE = E_T3 - E_R3
    dE_per_node = dE / N

    if verbose:
        print(f'  N={N}, g₄={g4}:  E_T³={E_T3:.3f}, E_R³={E_R3:.3f}, ΔE={dE:.3f}')
        print(f'    χ_H(T³)={chi_T3:.4f}  χ_H(R³)={chi_R3:.4f}')
        print(f'    iter T³={it_T3}  iter R³={it_R3}')

    return dict(N=N, N_side=N_side, g4=g4, E_T3=E_T3, E_R3=E_R3,
                dE=dE, dE_per_node=dE_per_node,
                chi_T3=chi_T3, chi_R3=chi_R3,
                iter_T3=it_T3, iter_R3=it_R3)

---## 5. Barrido principalEjecutar las 20 mediciones (5 g₄ × 4 N). Cada una toma ~1–10 minutos según N.**Persistencia:** se guarda cada punto a medida que se calcula en `sim21_progress.json`. Si Colab se desconecta y volvés a correr esta celda, retoma desde donde quedó.

In [ ]:
SIM21_FILE = out_path('sim21_progress.json')

# Cargar progreso si existe
if os.path.exists(SIM21_FILE):
    with open(SIM21_FILE) as f:
        results = json.load(f)
    print(f'Cargado progreso previo: {len(results)} puntos calculados')
else:
    results = []
    print('Iniciando barrido desde cero')

# Identificar puntos faltantes
ya_hechos = set((r['N_side'], r['g4']) for r in results)
faltantes = [(N_side, g4) for N_side in N_SIDES for g4 in G4_VALS
             if (N_side, g4) not in ya_hechos]
total_puntos = len(N_SIDES) * len(G4_VALS)
print(f'Puntos totales: {total_puntos}')
print(f'Puntos faltantes: {len(faltantes)}')

# Ejecutar faltantes
t_start = time.time()
for k, (N_side, g4) in enumerate(faltantes):
    N = N_side**3
    print(f'\n[{k+1}/{len(faltantes)}] N={N} (N_side={N_side}), g₄={g4} ...')
    t0 = time.time()
    pt = medir_punto(N_side, g4, verbose=True)
    dt = time.time() - t0
    print(f'    → tiempo: {dt:.1f}s')
    results.append(pt)
    # Guardar progreso
    with open(SIM21_FILE, 'w') as f:
        json.dump(results, f, indent=1)

print(f'\nTotal: {len(results)} puntos en {time.time()-t_start:.1f}s')

---## 6. Análisis: β(g₄) y w_top(g₄)Para cada g₄ ajustamos ΔE ∝ V^β (con V = N en este setup, a L = 1 fijo).Si la red tuviera spacing constante y la caja se expandiera con N, V ≈ N · L₀³ donde L₀ es el spacing; en nuestro setup la caja es L = 1 fija y N varía, entonces "volumen efectivo" es V = N (proporcional a la cantidad de modos en la región mesoscópica).La relación cosmológica entre β y w_top: si ΔE ∝ V^β, entonces ρ_top = ΔE/V ∝ V^(β−1). Comparando con la ecuación de continuidad cosmológica ρ ∝ a^(−3(1+w)) = V^(−(1+w)) (donde V = a³), obtenemos:**w_top = −β**(Esta convención asume que el sustrato escala uniformemente con el factor cosmológico. Es la convención de los Tests A/B del Cap. 6 §6.2; en el armónico ese cálculo dio +0,07 universal por la combinación de β = +0,99 con la corrección sub-líder del kernel 1/d² — acá nos enfocamos en cómo cambia β con g₄, no en el valor absoluto de w_top.)

In [ ]:
# Reorganizar resultados por g₄
import numpy as np
results = sorted(results, key=lambda r: (r['g4'], r['N']))

by_g4 = {}
for r in results:
    by_g4.setdefault(r['g4'], []).append(r)
for g4 in by_g4:
    by_g4[g4] = sorted(by_g4[g4], key=lambda r: r['N'])

# Ajuste log-log de ΔE vs N para cada g₄
print(f'{"g₄":>10s}  {"β":>10s}  {"R²":>8s}  {"χ_H max":>10s}  {"ΔE rango":>20s}')
print('─' * 70)

ajustes = {}
for g4 in sorted(by_g4.keys()):
    pts = by_g4[g4]
    Ns = np.array([p['N'] for p in pts])
    dEs = np.array([abs(p['dE']) for p in pts])  # |ΔE| para ajuste log
    chis = max(max(p['chi_T3'] for p in pts), max(p['chi_R3'] for p in pts))

    # log-log fit
    valid = dEs > 0
    if valid.sum() < 3:
        beta = np.nan; R2 = np.nan
    else:
        lN = np.log(Ns[valid]); ldE = np.log(dEs[valid])
        coef = np.polyfit(lN, ldE, 1)
        beta = coef[0]
        ldE_pred = np.polyval(coef, lN)
        ss_res = np.sum((ldE - ldE_pred)**2)
        ss_tot = np.sum((ldE - ldE.mean())**2)
        R2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    ajustes[g4] = dict(beta=beta, R2=R2, chi_max=chis, dEs=dEs, Ns=Ns,
                        w_top=-beta, signed_dE=np.array([p['dE'] for p in pts]))
    print(f'{g4:>10.1f}  {beta:>+10.4f}  {R2:>8.4f}  {chis:>10.4f}  '
          f'[{dEs.min():>8.2e}, {dEs.max():>8.2e}]')

In [ ]:
# Análisis específico: el control armónico (g₄=0) debe dar β ≈ +0,99
beta_armonico = ajustes[0.0]['beta']
print(f'\n═══ CONTROL ARMÓNICO (g₄ = 0) ═══')
print(f'  β medido = {beta_armonico:+.4f}')
print(f'  β esperado (Test A v5/v6 Cap. 6 §6.2) = +0,99')
diff_control = abs(beta_armonico - 0.99)
print(f'  Diferencia |β_med − 0,99| = {diff_control:.4f}')
print(f'  {"✓ Reproduce Test A" if diff_control < 0.20 else "⚠ NO reproduce Test A"}')

# Sensibilidad de β a g₄
print(f'\n═══ SENSIBILIDAD A g₄ ═══')
g4_max = max(ajustes.keys())
beta_max = ajustes[g4_max]['beta']
delta_beta = abs(beta_max - beta_armonico)
print(f'  β(g₄ = 0)        = {beta_armonico:+.4f}')
print(f'  β(g₄ = {g4_max:.0f}) = {beta_max:+.4f}')
print(f'  Δβ                = {delta_beta:.4f}')
print(f'  Criterio pre-registrado (Δβ > 0,1): {"✓ HAY DINÁMICA" if delta_beta > 0.1 else "✗ NO HAY DINÁMICA SIGNIFICATIVA"}')

# Veredicto
print(f'\n═══ VEREDICTO ═══')
if delta_beta > 0.1:
    if beta_max < beta_armonico:
        print(f'  H1 confirmada: β disminuye con g₄ (β_max < β_armónico)')
        print(f'  Esto implica w_top = −β AUMENTA con g₄')
        print(f'  → Componente potencial de w(z) en dirección w_a > 0')
    else:
        print(f'  H1 confirmada: β aumenta con g₄ (β_max > β_armónico)')
        print(f'  Esto implica w_top = −β DISMINUYE con g₄ (más negativo)')
        print(f'  → Componente potencial de w(z) en dirección w_a < 0 (compatible con DESI CPL evolutivo)')
elif delta_beta > 0.03:
    print(f'  H2 (mezcla intermedia): efecto pequeño pero no nulo')
    print(f'  Componente w_a podría ser detectable observacionalmente')
else:
    print(f'  H0 confirmada: topología NO contribuye dinámicamente en régimen anharmónico')
    print(f'  La predicción w constante del Cap. 6 §6.4 queda confirmada por este test')

---## 7. Visualización

In [ ]:
# Plot principal: ΔE vs N para cada g₄, log-log
fig, axes = new_figure(2, 2, figsize=(14, 10))
colors = plt.cm.viridis(np.linspace(0, 0.85, len(G4_VALS)))

# Panel A: log-log de |ΔE| vs N
ax1 = axes[0, 0]
for g4, col in zip(sorted(by_g4.keys()), colors):
    aj = ajustes[g4]
    ax1.loglog(aj['Ns'], aj['dEs'], 'o-', color=col, ms=9, lw=2,
               label=f'g₄={g4:.0f}  β={aj["beta"]:+.3f}')
ax1.set_xlabel('N (≈ V)', color=CW); ax1.set_ylabel('|ΔE| = |E_T³ − E_R³|', color=CW)
ax1.set_title('Diferencia topológica vs tamaño\nEscalado β(g₄)', color=CW, fontweight='bold')
ax1.legend(facecolor=BG, labelcolor=CW, fontsize=9, loc='best')
ax1.grid(True, alpha=0.15, which='both')

# Panel B: β(g₄)
ax2 = axes[0, 1]
g4s_plot = np.array(sorted(by_g4.keys()))
betas_plot = np.array([ajustes[g]['beta'] for g in g4s_plot])
R2s_plot = np.array([ajustes[g]['R2'] for g in g4s_plot])

ax2.semilogx(np.maximum(g4s_plot, 0.1), betas_plot, 'o-', color=CB, ms=12, lw=2.5,
             label='β medido')
ax2.axhline(0.99, color=CG, lw=2, ls='--', label='β armónico (Test A) = +0,99')
ax2.axhline(beta_armonico, color=CY, lw=2, ls=':',
            label=f'β(g₄=0) medido = {beta_armonico:+.3f}')
for g, b, r in zip(g4s_plot, betas_plot, R2s_plot):
    ax2.annotate(f'R²={r:.3f}', xy=(max(g, 0.1), b), xytext=(5, 5),
                  textcoords='offset points', fontsize=7.5, color=CGR)
ax2.set_xlabel('g₄ (0 → armónico)', color=CW); ax2.set_ylabel('β (exponente ΔE ∝ N^β)', color=CW)
ax2.set_title('Sensibilidad de β al régimen anharmónico', color=CW, fontweight='bold')
ax2.legend(facecolor=BG, labelcolor=CW, fontsize=9)
ax2.grid(True, alpha=0.15)

# Panel C: w_top(g₄) = -β(g₄)
ax3 = axes[1, 0]
w_tops = -betas_plot
ax3.semilogx(np.maximum(g4s_plot, 0.1), w_tops, 'o-', color=CR, ms=12, lw=2.5,
             label='w_top = −β')
ax3.axhline(-1, color='white', lw=1.5, ls='--', alpha=0.5, label='ΛCDM: w = −1')
ax3.axhline(-2/3, color=CP, lw=1.5, ls=':', label='Frontera holográfica: w = −2/3')
ax3.axhline(-0.70, color=CG, lw=2, ls='--',
            label='Cap. 6 §6.4: w_DE = −0,70 (homogéneo anharm.)')
ax3.set_xlabel('g₄', color=CW); ax3.set_ylabel('w_top derivado de β', color=CW)
ax3.set_title('Contribución cosmológica del sector topológico\n(referencia: w_DE central del modelo)',
              color=CW, fontweight='bold')
ax3.legend(facecolor=BG, labelcolor=CW, fontsize=8.5)
ax3.grid(True, alpha=0.15)

# Panel D: χ_Hartree(g₄) para validar régimen perturbativo
ax4 = axes[1, 1]
chi_max_plot = np.array([ajustes[g]['chi_max'] for g in g4s_plot])
ax4.loglog(np.maximum(g4s_plot, 0.1), np.maximum(chi_max_plot, 1e-6), 'o-',
           color=CO, ms=12, lw=2.5, label='χ_Hartree máx (T³ y R³)')
ax4.axhline(0.5, color=CR, lw=2, ls='--', label='límite régimen perturbativo (0,5)')
ax4.axhline(0.1, color=CG, lw=1.5, ls=':', label='Cap. 6 plateau (χ_H < 0,1)')
ax4.set_xlabel('g₄', color=CW); ax4.set_ylabel('χ_Hartree', color=CW)
ax4.set_title('Validez del régimen Hartree autoconsistente', color=CW, fontweight='bold')
ax4.legend(facecolor=BG, labelcolor=CW, fontsize=9)
ax4.grid(True, alpha=0.15, which='both')

fig.suptitle(
    f'SIM 21 — Topología anharmónica  →  ¿w(z) dinámico?\n'
    f'Δβ = {delta_beta:.3f}  (criterio Δβ > 0,1)  '
    f'β(g₄=0) = {beta_armonico:+.3f}  vs Test A esperado +0,99',
    fontsize=11, fontweight='bold', color=CW)
plt.tight_layout()
plt.savefig(out_path('sim21_topologia_anharmonica.png'), dpi=150,
            bbox_inches='tight', facecolor='#0a0a1a')
plt.show()

In [ ]:
# Plot adicional: signo y magnitud de ΔE en escala lineal
fig, ax = new_figure(1, 1, figsize=(10, 6))
for g4, col in zip(sorted(by_g4.keys()), colors):
    aj = ajustes[g4]
    ax.plot(aj['Ns'], aj['signed_dE'], 'o-', color=col, ms=9, lw=2,
            label=f'g₄ = {g4:.0f}')
ax.axhline(0, color=CW, lw=1, ls='--', alpha=0.5)
ax.set_xlabel('N', color=CW); ax.set_ylabel('ΔE = E_T³ − E_R³ (con signo)', color=CW)
ax.set_title('ΔE topológica con signo: ¿el régimen anharmónico cambia el sentido?',
             color=CW, fontweight='bold')
ax.legend(facecolor=BG, labelcolor=CW, fontsize=10)
ax.grid(True, alpha=0.15)
fig.suptitle('SIM 21 — Magnitud y signo de la diferencia topológica',
             fontsize=11, fontweight='bold', color=CW)
plt.tight_layout()
plt.savefig(out_path('sim21_signed_dE.png'), dpi=150, bbox_inches='tight', facecolor='#0a0a1a')
plt.show()

---## 8. Resumen ejecutivo y consolidado

In [ ]:
# Tabla consolidada
print('═' * 78)
print('  SIM 21 — Topología anharmónica: resumen consolidado'.center(78))
print('═' * 78)
print()
print(f'{"g₄":>10s}  {"β":>10s}  {"R²":>8s}  {"w_top = −β":>12s}  {"χ_H max":>10s}  {"Veredicto":>20s}')
print('─' * 78)
for g4 in sorted(by_g4.keys()):
    aj = ajustes[g4]
    if g4 == 0:
        v = 'control armónico'
    elif aj['chi_max'] > 0.5:
        v = 'fuera Hartree'
    elif abs(aj['beta'] - beta_armonico) > 0.1:
        v = 'dinámica'
    elif abs(aj['beta'] - beta_armonico) > 0.03:
        v = 'mezcla intermedia'
    else:
        v = 'sin cambio'
    print(f'{g4:>10.1f}  {aj["beta"]:>+10.4f}  {aj["R2"]:>8.4f}  '
          f'{-aj["beta"]:>+12.4f}  {aj["chi_max"]:>10.4f}  {v:>20s}')

print('─' * 78)
print()
print('CONEXIÓN CON EL DOCUMENTO DEE v6:')
print()
print(f'  • Control g₄=0: β = {beta_armonico:+.3f}')
print(f'    Test A v5/v6 original esperaba β ≈ +0,99')
print(f'    Diferencia: {abs(beta_armonico - 0.99):.3f} → '
      f'{"reproduce ✓" if abs(beta_armonico - 0.99) < 0.20 else "no reproduce ⚠"}')
print()
print(f'  • Δβ entre g₄=0 y g₄_max = {delta_beta:.3f}')
print(f'    Criterio pre-registrado: Δβ > 0,1 → {"DINÁMICA" if delta_beta > 0.1 else "NO DINÁMICA"}')
print()
print(f'  • Implicación para w_DE:')
if delta_beta > 0.1:
    print(f'    El sector topológico anharmónico responde al régimen no-cuadrático.')
    print(f'    La predicción w = −0,70 ± 0,05 del Cap. 6 §6.4 (homogéneo anharmónico)')
    print(f'    se complementa con una contribución topológica de signo y magnitud:')
    print(f'    Δw_top ≈ {-delta_beta:+.3f}  con escalado g₄-dependiente.')
    print(f'    Esto abre la posibilidad de w(z) dinámico con w_a determinado por g₄·V⁻ⁿ.')
    print(f'    Alineado con DESI 2024 CPL evolutivo si la dirección de w_a coincide.')
else:
    print(f'    El sector topológico NO contribuye dinámicamente en el rango g₄ ∈ [0, {max(G4_VALS):.0f}].')
    print(f'    La predicción w constante del Cap. 6 §6.4 queda confirmada por exclusión.')
    print(f'    Si DESI DR3 confirma w_a no compatible con cero, será necesario buscar')
    print(f'    el mecanismo de dinámica en otra dirección (no en topología + anharmonicidad).')

print()
print('Archivos generados:')
for f in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f))
    print(f'  {f}  ({sz/1024:.1f} KB)')

In [ ]:
# Empaquetar para descarga
import subprocess
subprocess.run(['zip', '-q', '-r', os.path.join(OUT_DIR, 'sim21_resultados.zip'),
                '.'], cwd=OUT_DIR)
print(f'\nDescarga:  {OUT_DIR}/sim21_resultados.zip')

---## Qué hacer con los resultados1. Descargar `sim21_resultados.zip` (panel izquierdo de Colab)2. Pasarme:   - El `.png` de los cuatro paneles   - La salida de texto del resumen consolidado   - Si querés, el `sim21_progress.json` con los números crudos3. Con eso decidimos cómo va al documento v6:   - **Si Δβ > 0,1 (dinámica confirmada):** se incorpora como SIM 21 en Cap. 6 §6.3 con su propia subsección, la predicción central se refina a w(z) con componente w_a derivable, y el Cap. 7 §7.3 cambia el caveat sobre DESI CPL por una predicción positiva refinada.   - **Si Δβ < 0,03 (no dinámica):** se incorpora como nota cuantitativa al final del Cap. 6 §6.2 confirmando que también el sector topológico anharmónico produce w constante. El compromiso de falsabilidad del Cap. 7 §7.3 queda como está.   - **Si Δβ ∈ [0,03, 0,1] (mezcla intermedia):** se reporta como predicción positiva potencial con magnitud sub-líder.**Si algo falla:**- Hartree no converge para g₄ grande → bajar `g₄_max` a 5000 o aumentar `HARTREE_MAX` a 40- Memoria insuficiente para N=2197 → quitar N_side=13 de la lista- Cálculo demasiado lento → reducir K_NEIGHBORS de 30 a 20